# 1.1 downdepth for annotation rds to .rds

In [ ]:
library(Seurat)
library(DropletUtils)

# choose SA_5 as sample
input_rds <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/8sample_rds/SA_5.rds"
seurat_obj <- readRDS(input_rds)
counts <- GetAssayData(seurat_obj, assay="RNA", layer = "counts")

out_root <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"

scales <- c(0.25, 0.5, 0.75)
n_repeats <- 3

for (p in scales) {
  for (r in 1:n_repeats) {

    message("Downsampling ", p*100, "% repeat ", r, " .\n")

    outdir <- file.path(out_root, 
                        paste0("SA5_", p*100, "_percentage"), 
                        paste0("repeat_", r), "annotated_outs")
    dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

    ds_counts <- downsampleMatrix(counts, prop = p)

    ds_obj <- CreateSeuratObject(ds_counts)

    barcodes_ds <- colnames(ds_obj)
    ds_obj$celltype <- seurat_obj$celltype[barcodes_ds]

    saveRDS(ds_obj, file = file.path(outdir, "anno_ds.rds"))
    message("Done.\n")
  }
}


# 1.2 downdepth for cellranger outs/raw and outs/filtered to .rds
Since some methods calculate contamination ratio and decontamination matrix by cellranger outs results

In [ ]:
library(Seurat)
library(DropletUtils)
library(Matrix)

raw_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/01_rawdata/GSE157100/GSM4752989/cellranger/GSM4752989/outs/raw_feature_bc_matrix"
filtered_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/01_rawdata/GSE157100/GSM4752989/cellranger/GSM4752989/outs/filtered_feature_bc_matrix"

message("Load data.\n")
raw_counts <- Read10X(raw_dir)
filtered_counts <- Read10X(filtered_dir)
message("Done.\n")
 
scales <- c(0.25, 0.5, 0.75)
n_repeats <- 3

for (p in scales) {
  for (r in 1:n_repeats) {

    message("Downsampling ", p*100, "% repeat ", r, " .\n")

    outdir <- paste0("/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth/SA5_", 
                     p*100, "_percentage/repeat_", r,"/cellranger_outs")
    dir.create(outdir, recursive = TRUE, showWarnings = FALSE)
    
    # raw counts downdepth
    ds_raw <- downsampleMatrix(raw_counts, prop = p)
    ds_raw_obj <- CreateSeuratObject(ds_raw)
    saveRDS(ds_raw_obj, file = file.path(outdir, paste0("raw_ds.rds")))
    
    # filtered counts downdepth
    ds_filtered <- downsampleMatrix(filtered_counts, prop = p)
    ds_filtered_obj <- CreateSeuratObject(ds_filtered)
    saveRDS(ds_filtered_obj, file = file.path(outdir, paste0("filtered_ds.rds")))
    
    message("Done.\n")
  }
}


# 2. Generate data from seurat object (.rds)

This first data is considered as 'rawdata' with different sequencing depth (25%, 50%, 75%), which is used to calculate ground-truth contamination ratio and so on. The results are saved to 'annotated_outs' folder.

This second data is curated from cellranger outs, which is used for each method. The results are saved to 'cellranger_outs' folder.

## Helper function

In [ ]:

library(Seurat)
library(DropletUtils)
library(sceasy)
library(Matrix)

export_downsampled_10x <- function(seurat_obj, outdir) {
  mat <- GetAssayData(seurat_obj, assay = "RNA", layer = "counts")
  barcodes <- colnames(mat)
  features <- rownames(mat)

  dir.create(outdir, showWarnings = FALSE, recursive = TRUE)

  # 1. trans annotated seurat object rds to cellrange out format folder
  mtx_dir <- file.path(outdir, "filtered_feature_bc_matrix")
  dir.create(mtx_dir, showWarnings = FALSE)

  writeMM(mat, file.path(mtx_dir, "matrix.mtx"))
  R.utils::gzip(file.path(mtx_dir, "matrix.mtx"), overwrite = TRUE)

  write.table(barcodes, file.path(mtx_dir, "barcodes.tsv"),
              quote = FALSE, row.names = FALSE, col.names = FALSE)
  R.utils::gzip(file.path(mtx_dir, "barcodes.tsv"), overwrite = TRUE)

  write.table(
    cbind(features, features, "Gene Expression"),
    file.path(mtx_dir, "features.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE, sep = "\t"
  )
  R.utils::gzip(file.path(mtx_dir, "features.tsv"), overwrite = TRUE)

  # 2. trans annotated seurat object rds to 10x h5 data, same as cellranger outs format
  DropletUtils::write10xCounts(
    file.path(outdir, "filtered_feature_bc_matrix.h5"),
    mat,
    gene.id = features,
    gene.symbol = features,
    overwrite = TRUE,
    version = "3"   # corresopnding to cellranger v3+
  )

  # 3. trans annotated seurat object rds to h5ad data
  sceasy::convertFormat(seurat_obj, from = "seurat", to = "anndata", outFile = file.path(outdir, "filtered_feature_bc_matrix.h5ad"))

  message("Export finished: ", outdir)
}

## 2.1 annotated  data

In [ ]:

base_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"

percentages <- c("SA5_25_percentage", "SA5_50_percentage","SA5_75_percentage")
repeats <- paste0("repeat_", 1:3)

for (pct in percentages) {
  for (rep in repeats) {

    rep_dir <- file.path(base_dir, pct, rep, "annotated_outs")
    ds_rds <- file.path(rep_dir, "anno_ds.rds")

    if (file.exists(ds_rds)) {
      message("Exporting: ", ds_rds)
      seurat_obj <- readRDS(ds_rds)
      export_downsampled_10x(seurat_obj = seurat_obj, outdir = rep_dir)
    }
  }
}

## 2.2 cellranger outs data

In [ ]:

base_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"

percentages <- c("SA5_25_percentage", "SA5_50_percentage","SA5_75_percentage")
repeats <- paste0("repeat_", 1:3)

for (pct in percentages) {
  for (rep in repeats) {

    rep_dir <- file.path(base_dir, pct, rep, "cellranger_outs")

    # --- raw ---
    raw_rds <- file.path(rep_dir, "raw_ds.rds")
    if (file.exists(raw_rds)) {
      message("Exporting RAW: ", raw_rds)
      seurat_obj <- readRDS(raw_rds)
      export_downsampled_10x(
        seurat_obj = seurat_obj,
        outdir = file.path(rep_dir, "raw")
      )
    }

    # --- filtered ---
    filtered_rds <- file.path(rep_dir, "filtered_ds.rds")
    if (file.exists(filtered_rds)) {
      message("Exporting FILTERED: ", filtered_rds)
      seurat_obj <- readRDS(filtered_rds)
      export_downsampled_10x(
        seurat_obj = seurat_obj,
        outdir = file.path(rep_dir, "filtered")
      )
    }

  }
}

# 3. Calculate contamination ratio and decontamination (corrected) matrix by different methods

## CellBender

conda activate cb; 150 epochs

1. run cellbender using bash cli, require raw h5 file we generated from cellranger outs by decresing depth

In [ ]:
#!/bin/bash
scales=(25 50 75)
repeats=(1 2 3)

root_dir="/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"

for scale in "${scales[@]}"; do
    for rep in "${repeats[@]}"; do
        input="${root_dir}/SA5_${scale}_percentage/repeat_${rep}/cellranger_outs/raw/filtered_feature_bc_matrix.h5"
        output_dir="${root_dir}/SA5_${scale}_percentage/repeat_${rep}/cellbender"
        mkdir -p "$output_dir"
        output="${output_dir}/cellbender_output_file.h5"
        
        echo "Running CellBender for SA5_${scale}%, repeat ${rep} ..."
        cellbender remove-background \
            --cuda \
            --input "$input" \
            --output "$output"
    done
done


2. load results, require "cellbender_output_file.h5" rather "cellbender_output_file_filtered.h5", then use background_fraction > 0.5 to filter, finally obtain contam ratio per cell.


In [ ]:
library(Seurat)
library(rhdf5)

root_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"
scales <- c(25, 50, 75)
repeats <- 1:3

for (scale in scales) {
  for (rep in repeats) {
    message("Processing SA5_", scale, "%, repeat ", rep)

    # 1. Read the downsampled Seurat object
    rds_file <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                          paste0("repeat_", rep), "annotated_outs", "anno_ds.rds")
    seurat_obj <- readRDS(rds_file)

    # 2. Extract OSN/non-OSN  
    seurat_obj <- SetIdent(seurat_obj, value = "celltype")
    non_osn_cells <- subset(seurat_obj, idents = "Mature OSNs", invert = TRUE)
    osn_cells <- subset(seurat_obj, idents = "Mature OSNs")

    # Obtain the counts matrix
    non_osn_cells_counts <- GetAssayData(non_osn_cells[["RNA"]], layer = "counts")
    osn_cells_counts <- GetAssayData(osn_cells[["RNA"]], layer = "counts")

    # 3. Read the output of CellBender
    result_file <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                             paste0("repeat_", rep), "cellbender", "cellbender_output_file.h5")

    metadata <- h5read(result_file, "/metadata")
    all_barcode <- metadata$barcodes_analyzed

    droplet_latents <- h5read(result_file, "/droplet_latents")
    contamination_ratio <- data.frame(
      background_fraction = droplet_latents$background_fraction,
      cell_probability = droplet_latents$cell_probability
    )
    rownames(contamination_ratio) <- all_barcode
    contamination_ratio_filtered <- contamination_ratio[contamination_ratio$cell_probability > 0.5, ]

    # 4. Calculate per-cell contamination
    get_contamination_rate <- function(cells, contamination_df) {
      barcodes <- colnames(cells)
      vals <- contamination_df[barcodes, "background_fraction"]
      names(vals) <- barcodes
      return(vals)
    }

    cellbender_non_osn <- get_contamination_rate(non_osn_cells_counts, contamination_ratio_filtered)
    cellbender_osn <- get_contamination_rate(osn_cells_counts, contamination_ratio_filtered)

    df_non_osn <- data.frame(
      barcodes = names(cellbender_non_osn),
      cellbender = as.numeric(cellbender_non_osn),
      celltype = "nonOSN",
      stringsAsFactors = FALSE
    )
    df_osn <- data.frame(
      barcodes = names(cellbender_osn),
      cellbender = as.numeric(cellbender_osn),
      celltype = "OSN",
      stringsAsFactors = FALSE
    )

    df_cellbender <- rbind(df_osn, df_non_osn)

    out_file <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                          paste0("repeat_", rep), "cellbender_contamination_rate_percell.csv")

    write.csv(df_cellbender, out_file, row.names = FALSE)
    message("Saved contamination rates to ", out_file)
  }
}


3. generate corrected matrix (.h5ad)

In [ ]:
import scanpy as sc

scales = [25, 50, 75]
repeats = [1, 2, 3]

for scale in scales:
    for rep in repeats:
        # Input path (.h5 file)
        input_path = f"/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth/SA5_{scale}_percentage/repeat_{rep}/cellbender/cellbender_output_file_filtered.h5"
        
        # Output path (.h5ad file, save to different locations)
        output_path = f"/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth/SA5_{scale}_percentage/repeat_{rep}/cellbender/cellbender_output_file_filtered.h5ad"
        
        adata = sc.read_10x_h5(input_path)
        
        # Save in the.h5ad format
        adata.write(output_path)
        
        print(f"Finish: scale={scale}%, repeat={rep}")

## FastCAR

filtered_GEX: generated 10x .h5 filtered data with different sequecing depth

raw_GEX: generated 10x .h5 raw data with different sequecing depth


In [ ]:
library(FastCAR)
library(Seurat)
library(sceasy)
library(Matrix)

root_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"
scales <- c(25, 50, 75)
repeats <- 1:3

for (scale in scales) {
  for (rep in repeats) {
    message("Processing FastCAR: ", scale, "%, repeat ", rep)

    # input raw/filtered h5
    raw_h5 <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                        paste0("repeat_", rep), "cellranger_outs/raw/filtered_feature_bc_matrix.h5")
    filtered_h5 <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                             paste0("repeat_", rep), "cellranger_outs/filtered/filtered_feature_bc_matrix.h5")

    raw_GEX <- Read10X_h5(raw_h5)
    filtered_GEX <- Read10X_h5(filtered_h5)

    # load down depth seurat object to get OSN/non-OSN counts
    seurat_rds <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                            paste0("repeat_", rep), "annotated_outs/anno_ds.rds")
    seurat_obj <- readRDS(seurat_rds)
    seurat_obj <- SetIdent(seurat_obj, value = "celltype")
    
    non_osn_cells <- subset(seurat_obj, idents = "Mature OSNs", invert = TRUE)
    osn_cells <- subset(seurat_obj, idents = "Mature OSNs")

    non_osn_cells_counts <- GetAssayData(non_osn_cells[["RNA"]], layer = "counts")
    osn_cells_counts <- GetAssayData(osn_cells[["RNA"]], layer = "counts")

    # FastCAR ambient RNA
    ambProfile <- describe.ambient.RNA.sequence(
      fullCellMatrix = raw_GEX,
      start = 10, stop = 500, by = 10,
      contaminationChanceCutoff = 0.05
    )
    emptyDropletCutoff <- recommend.empty.cutoff(ambProfile)
    contaminationChanceCutoff <- 0.05
    
    ambientProfile <- determine.background.to.remove(raw_GEX, emptyDropletCutoff, contaminationChanceCutoff)
    ambient_genes <- names(ambientProfile[ambientProfile > 0])

    # per-cell contamination rate
    if(length(ambient_genes) <= 1) {
        fastcar_non_osn <- non_osn_cells_counts[ambient_genes, ] / colSums(non_osn_cells_counts)
        fastcar_osn <- osn_cells_counts[ambient_genes, ] / colSums(osn_cells_counts)
    } else {
        fastcar_non_osn <- colSums(non_osn_cells_counts[ambient_genes, ]) / colSums(non_osn_cells_counts)
        fastcar_osn <- colSums(osn_cells_counts[ambient_genes, ]) / colSums(osn_cells_counts)
    }

    df_non_osn <- data.frame(barcodes = names(fastcar_non_osn),
                             fastcar = as.numeric(fastcar_non_osn),
                             celltype = "nonOSN", stringsAsFactors = FALSE)
    df_osn <- data.frame(barcodes = names(fastcar_osn),
                         fastcar = as.numeric(fastcar_osn),
                         celltype = "OSN", stringsAsFactors = FALSE)
    df_fastcar <- rbind(df_osn, df_non_osn)

    # save per-cell contamination rate
    out_csv <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                         paste0("repeat_", rep), "fastcar_contamination_rate_percell.csv")
    write.csv(df_fastcar, out_csv, row.names = FALSE)
    message("Saved FastCAR contamination rates to ", out_csv)

    # save corrected matrix
    fastcar_cellMatrix <- remove.background(filtered_GEX, ambientProfile)
    fastcar_seurat_obj <- CreateSeuratObject(fastcar_cellMatrix)

    out_h5ad <- file.path(root_dir, paste0("SA5_", scale, "_percentage"),
                          paste0("repeat_", rep), "fastcar/fastcar_corrected_mat.h5ad")
    dir.create(dirname(out_h5ad), recursive = TRUE, showWarnings = FALSE)
    sceasy::convertFormat(fastcar_seurat_obj, from = "seurat", to = "anndata", outFile = out_h5ad)
    message("Saved FastCAR corrected h5ad to ", out_h5ad)
  }
}


## scAR

conda activate scar 

1. use cellranger out filtered h5 file as input, run bash cli

In [ ]:
#!/bin/bash

scales=(25 50 75)
repeats=(1 2 3)

root_dir="/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"

for scale in "${scales[@]}"; do
    for rep in "${repeats[@]}"; do
        input="${root_dir}/SA5_${scale}_percentage/repeat_${rep}/cellranger_outs/filtered/filtered_feature_bc_matrix.h5"
        output_dir="${root_dir}/SA5_${scale}_percentage/repeat_${rep}/scar"
        mkdir -p "$output_dir"

        echo "Running SCAR for SA5_${scale}%, repeat ${rep} ..."
        scar "$input" \
            -ft mRNA \
            -o "$output_dir" \
            -d cuda
    done
done

2.calculate contamination rate

In Cd36_OSNs env

In [ ]:
library(Seurat)
library(reticulate)
use_condaenv("/data/R04/zhangchao/anaconda3/envs/scar", required = TRUE)
py_config()
sc <- import("scanpy")

root_dir <- "/data/R04/zhangchao/joint_nuclei/03_analysis"
scales <- c(25, 50, 75)
repeats <- 1:3

for (scale in scales) {
  for (rep in repeats) {
    message("Processing SCAR contamination: ", scale, "%, repeat ", rep)

    # 1. Read the Seurat downsampled object
    rds_file <- file.path(root_dir, "SA5_downdepth", paste0("SA5_", scale, "_percentage/", "repeat_", rep),
                          "annotated_outs", "anno_ds.rds")
    seurat_obj <- readRDS(rds_file)
    seurat_obj <- SetIdent(seurat_obj, value = "celltype")
    non_osn_cells <- subset(seurat_obj, idents = "Mature OSNs", invert = TRUE)
    osn_cells <- subset(seurat_obj, idents = "Mature OSNs")

    # 2. Read the SCAR denoised h5ad file
    h5ad_file <- file.path(root_dir, "SA5_downdepth", paste0("SA5_", scale, "_percentage/", "repeat_", rep),
                           "scar", "filtered_feature_bc_matrix_denoised_mRNA.h5ad")
    
    adata <- sc$read_h5ad(h5ad_file)
    
    obs_names <- rownames(adata$obs)
    noise_ratio <- adata$obs$noise_ratio
    names(noise_ratio) <- obs_names

    all_contam <- noise_ratio


    # 3. Extract contamination ratio by cell type
    scar_non_osn <- all_contam[colnames(non_osn_cells)]
    scar_osn <- all_contam[colnames(osn_cells)]

    # 4. Save per-cell contamination
    df_non_osn <- data.frame(
      barcodes = names(scar_non_osn),
      scar = as.numeric(scar_non_osn),
      celltype = "nonOSN",
      stringsAsFactors = FALSE
    )
    df_osn <- data.frame(
      barcodes = names(scar_osn),
      scar = as.numeric(scar_osn),
      celltype = "OSN",
      stringsAsFactors = FALSE
    )
    df_scar <- rbind(df_osn, df_non_osn)

    out_file <- file.path(root_dir, "SA5_downdepth", paste0("SA5_", scale, "_percentage/", "repeat_", rep),
                          "scar_contamination_rate_percell.csv")
    write.csv(df_scar, out_file, row.names = FALSE)
    message("Saved SCAR contamination rates to ", out_file)
  }
}


# scCDC

get corrected matrix from rds

In [ ]:
library(Seurat)

scales <- c(25, 50, 75)
repeats <- 1:3
root_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/depth"

for (scale in scales) {
  for (rep in repeats) {
    path = file.path(root_dir,"corrected_rds", paste0("scCDC_SA5_",scale,"_rep",rep,".rds"))
    seurat_obj <- readRDS(path)

    counts_mat <- GetAssayData(seurat_obj, assay = "RNA", layer = "counts") # v5 style

    seurat_obj <- CreateSeuratObject(counts_mat)

    output_path <- file.path(
      "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth", 
      paste0("SA5_", scale, "_percentage"), 
      paste0("repeat_", rep),
      paste0("scCDC/scCDC_",scale,"pct_corrected_mat.h5ad")
      )
    dir.create(dirname(output_path), recursive = TRUE, showWarnings = FALSE)

    sceasy::convertFormat(
        seurat_obj, 
        from = "seurat", 
        to = "anndata", 
        outFile = output_path, 
        main_layer = "counts"
    )
    message("Saved scCDC corrected h5ad to ", output_path)

  }
}


# Cellclear

copy correted matrix folder

In [ ]:
import os
import shutil
import re

src_root = "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/depth"
dst_root = "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth"

pattern = re.compile(r"(\d+)_cellclear_rep(\d+)_matrix")

for name in os.listdir(src_root):
    m = pattern.match(name)
    if not m:
        continue

    percent = m.group(1)   # 75
    rep = m.group(2)       # 1

    # Source matrix folder
    src_matrix = os.path.join(src_root, name, "matrix")
    if not os.path.isdir(src_matrix):
        print(f"Skip: no matrix folder in {name}")
        continue

    # target directory
    dst_dir = os.path.join(
        dst_root,
        f"SA5_{percent}_percentage",
        f"repeat_{rep}",
        "cellclear"
    )
    os.makedirs(dst_dir, exist_ok=True)

    # Target matrix folder
    dst_matrix = os.path.join(dst_dir, "matrix")

    # If it already exists, delete it first
    if os.path.exists(dst_matrix):
        shutil.rmtree(dst_matrix)

    # copy
    shutil.copytree(src_matrix, dst_matrix)

    print(f"Copied {src_matrix} -> {dst_matrix}")

print("Done.")


# DecontX

get corrected matrix from rds

In [ ]:
library(Seurat)

scales <- c(25, 50, 75)
repeats <- 1:3
root_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/depth"

for (scale in scales) {
  for (rep in repeats) {
    path = file.path(root_dir,"corrected_rds", paste0("decontx_SA5_",scale,"_rep",rep,".rds"))
    counts_mat <- readRDS(path)
    seurat_obj <- CreateSeuratObject(counts_mat)

    output_path <- file.path(
      "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth", 
      paste0("SA5_", scale, "_percentage"), 
      paste0("repeat_", rep),
      paste0("decontx/decontx_",scale,"pct_corrected_mat.h5ad")
    )
    dir.create(dirname(output_path), recursive = TRUE, showWarnings = FALSE)

    sceasy::convertFormat(
        seurat_obj, 
        from = "seurat", 
        to = "anndata", 
        outFile = output_path, 
        main_layer = "counts"
    )
    message("Saved decontx corrected h5ad to ", output_path)

  }
}


# soupx

get corrected matrix from rds

In [ ]:
library(Seurat)

scales <- c(25, 50, 75)
repeats <- 1:3
root_dir <- "/data/R02/tanj93/project/OSN_AmbientRNA_Benchmarking/20_8samples/depth"

for (scale in scales) {
  for (rep in repeats) {
    path = file.path(root_dir,"corrected_rds", paste0("SA5_",scale,"_rep",rep,"_soupx_corrected_counts.rds"))
    counts_mat <- readRDS(path)
    seurat_obj <- CreateSeuratObject(counts_mat)

    output_path <- file.path(
      "/data/R04/zhangchao/joint_nuclei/03_analysis/SA5_downdepth", 
      paste0("SA5_", scale, "_percentage"), 
      paste0("repeat_", rep),
      paste0("soupx/soupx_",scale,"pct_corrected_mat.h5ad")
    )
    dir.create(dirname(output_path), recursive = TRUE, showWarnings = FALSE)

    sceasy::convertFormat(
        seurat_obj, 
        from = "seurat", 
        to = "anndata", 
        outFile = output_path, 
        main_layer = "counts"
    )
    message("Saved soupx corrected h5ad to ", output_path)

  }
}
